# XML And Operator Serialization

This notebook is the seventh step in the operator sequence. The earlier notebooks focused on building and understanding operator pipelines. This one focuses on persistence: how to register operators, serialize a pipeline to XML, inspect the XML, and reconstruct an executable pipeline from it.

Previous: [06 Control Flow And Conditionals](./example_abstract_graph_operators_06_control_flow_and_conditionals.ipynb)  
Next: [08 Vectorization And Features](./example_abstract_graph_operators_08_vectorization_and_features.ipynb)


## Why XML matters here

An operator pipeline is a program over graphs. Once you can serialize it, you can store it, version it, compare it, send it across repositories, and reload it later without rebuilding the pipeline manually in Python.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import networkx as nx
from IPython.core.display import HTML
from warnings import simplefilter

simplefilter(action='ignore', category=FutureWarning)
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')


In [ ]:
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.display import display, display_decomposition_graph, display_graph, display_mappings
from abstractgraph.operators import *
from abstractgraph.xml import (
    register_from_module,
    operator_to_xml_string,
    operator_from_xml_string,
    operator_to_xml_file,
    operator_from_xml_file,
)
import abstractgraph.operators as ag_ops

register_from_module(ag_ops)


def draw(graph, decomposition_function, *, nbits=11, size=(12, 6), n_elements_per_row=8):
    display_decomposition_graph(decomposition_function)
    ag = graph_to_abstract_graph(graph, decomposition_function=decomposition_function, nbits=nbits)
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag


## The same toy graph

We keep using the same graph so it is easy to compare the round-tripped pipelines with earlier notebooks.


In [ ]:
graph = nx.Graph()
graph.add_nodes_from([
    (0, {'label': 'C'}),
    (1, {'label': 'C'}),
    (2, {'label': 'O'}),
    (3, {'label': 'N'}),
    (4, {'label': 'C'}),
    (5, {'label': 'S'}),
    (6, {'label': 'C'}),
    (7, {'label': 'O'}),
])
graph.add_edges_from([
    (0, 1, {'label': 'ring'}),
    (1, 2, {'label': 'ring'}),
    (2, 3, {'label': 'ring'}),
    (3, 0, {'label': 'ring'}),
    (2, 4, {'label': 'bridge'}),
    (4, 5, {'label': 'ring'}),
    (5, 2, {'label': 'ring'}),
    (3, 6, {'label': 'branch'}),
    (6, 7, {'label': 'branch'}),
])

print(graph)
display_graph(graph)


## Start with a simple pipeline

We begin with a small operator so the XML is easy to read.


In [ ]:
simple_df = compose(intersection_edges(), node())
simple_ag = draw(graph, simple_df)


## Serialize to XML

`operator_to_xml_string(...)` turns the operator pipeline into a textual representation. Because we already registered the operators from `abstractgraph.operators`, the serializer can use their names directly.


In [ ]:
simple_xml = operator_to_xml_string(simple_df, pretty=True)
print(simple_xml)


## Deserialize from XML

`operator_from_xml_string(...)` reconstructs a callable operator pipeline. We can run it exactly like the original one.


In [ ]:
simple_df_rt = operator_from_xml_string(simple_xml)
simple_rt_ag = draw(graph, simple_df_rt)


## Move to a richer pipeline

The next example adds filtering so the XML includes both composition structure and operator parameters.


In [ ]:
filter_df = compose(filter_by_node_label(must_have_one_of=['N']), cycle())
filter_ag = draw(graph, filter_df)


In [ ]:
filter_xml = operator_to_xml_string(filter_df, pretty=True)
print(filter_xml)


In [ ]:
filter_df_rt = operator_from_xml_string(filter_xml)
filter_rt_ag = draw(graph, filter_df_rt)


## Round-trip a `compose_product(...)` pipeline

This is the more important stress test. It shows that the XML layer can preserve a pipeline that branches into two decompositions and recombines them with a binary operator.


In [ ]:
product_df = compose(
    intersection_edges(),
    compose_product(binary_combination(distance=(0, 1)), cycle(), neighborhood(radius=1)),
)
product_ag = draw(graph, product_df)


In [ ]:
product_xml = operator_to_xml_string(product_df, pretty=True)
print(product_xml)


In [ ]:
product_df_rt = operator_from_xml_string(product_xml)
product_rt_ag = draw(graph, product_df_rt)


## Save XML to a file and load it back

String round-trips are useful inside Python, but file round-trips are what you typically want for reproducible workflows.


In [ ]:
xml_path = repo_root / 'notebooks' / 'examples' / 'tmp_operator_roundtrip.xml'
operator_to_xml_file(filter_df, str(xml_path), pretty=True)
print(xml_path)
print(xml_path.read_text())


In [ ]:
file_df_rt = operator_from_xml_file(str(xml_path))
file_rt_ag = draw(graph, file_df_rt)


## Cleanup

The XML file above is only for demonstration, so we remove it at the end of the notebook.


In [ ]:
xml_path.unlink(missing_ok=True)
print(xml_path.exists())


## Summary

At this point the operator sequence has covered decomposition, composition, filtering, binary operators, and XML persistence. The XML layer is what makes these pipelines portable and inspectable as structured programs rather than only live Python objects.

A natural next notebook is `07_vectorization_and_features.ipynb`, which would show how an `AbstractGraph` becomes a matrix representation for downstream ML workflows.

Previous: [06 Control Flow And Conditionals](./example_abstract_graph_operators_06_control_flow_and_conditionals.ipynb)  
Next: [08 Vectorization And Features](./example_abstract_graph_operators_08_vectorization_and_features.ipynb)
